# Connect Four AI 🔴🟡
Write the game, then an AI that **searches**: minimax looks ahead *d* moves assuming you play perfectly against it, and **alpha–beta pruning** skips branches that can't change the decision. Then play against it.

Background: [[RL Basics and MDPs]] (games as decision problems) · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import math, time, numpy as np
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

ROWS, COLS = 6, 7
ORDER = [3, 2, 4, 1, 5, 0, 6]                     # try centre columns first → much better pruning
def legal_moves(board): return [c for c in ORDER if board[0, c] == 0]
def drop(board, col, player):
    b = board.copy(); r = max(i for i in range(ROWS) if b[i, col] == 0); b[r, col] = player; return b
def show(board): print("\n".join(" ".join(".XO"[v] for v in row) for row in board) + "\n0 1 2 3 4 5 6")
def heuristic(board):
    """Score windows of 4: reward 2/3-in-a-row for player 1 (X), penalise for −1 (O), plus centre control."""
    s = 3 * (np.sum(board[:, 3] == 1) - np.sum(board[:, 3] == -1))
    for r in range(ROWS):
        for c in range(COLS):
            for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                cells = [(r + k * dr, c + k * dc) for k in range(4)]
                if all(0 <= i < ROWS and 0 <= j < COLS for i, j in cells):
                    w = [board[i, j] for i, j in cells]; p, o = w.count(1), w.count(-1)
                    if o == 0: s += {2: 2, 3: 10}.get(p, 0)
                    if p == 0: s -= {2: 2, 3: 10}.get(o, 0)
    return s

## 1. Who won?
Return `True` if `player` (1 or −1) has four in a row horizontally, vertically or on either diagonal.

In [ ]:
def wins(board, player):
    # TODO 1: check every start cell and the 4 directions
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_win():
    e = np.zeros((6, 7), int); h = e.copy(); h[5, 1:5] = 1; v = e.copy(); v[2:6, 0] = -1
    d = e.copy(); d[[5, 4, 3, 2], [0, 1, 2, 3]] = 1; a = e.copy(); a[[2, 3, 4, 5], [3, 4, 5, 6]] = 1; three = e.copy(); three[5, :3] = 1
    return wins(h, 1) and wins(v, -1) and not wins(v, 1) and wins(d, 1) and wins(a, 1) and not wins(three, 1)
check("wins", _t_win, "Horizontal, vertical and both diagonals must count; three in a row must not.")

## 2. Minimax with alpha–beta
X (player 1) maximises, O (−1) minimises. Return `(value, best_move)`. Terminal: a win is ±1,000,000 (± depth, so faster wins score higher); at depth 0 or a full board use `heuristic`. **Pruning:** stop exploring a node's children as soon as `alpha >= beta`.

In [ ]:
def minimax(board, depth, alpha=-math.inf, beta=math.inf, maximizing=True):
    # TODO 2: recursive minimax returning (value, move) with alpha-beta cut-offs
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_mm():
    e = np.zeros((6, 7), int)
    win_now = e.copy(); win_now[5, 0:3] = 1; win_now[4, 0:2] = -1; win_now[5, 6] = -1
    block = e.copy(); block[5, 3:6] = -1; block[5, 6] = 1; block[5, 0] = 1; block[4, 0] = 1
    return minimax(win_now, 2)[1] == 3 and minimax(block, 2)[1] == 2
check("minimax", _t_mm, "X must take the immediate win in column 3, and must block O's three in column 2.")
if ready("wins", "minimax"):
    rng = np.random.default_rng(0); results = []; t = time.time()
    for game in range(20):
        board, player = np.zeros((6, 7), int), 1
        while legal_moves(board) and not wins(board, 1) and not wins(board, -1):
            move = minimax(board, 3)[1] if player == 1 else rng.choice(legal_moves(board))
            board = drop(board, move, player); player = -player
        results.append(1 if wins(board, 1) else -1 if wins(board, -1) else 0)
    print(f"depth-3 AI vs random: {results.count(1)} wins, {results.count(-1)} losses, {results.count(0)} draws ({time.time()-t:.0f}s)")
    check("beats random", lambda: results.count(1) >= 19, "A depth-3 search should beat a random player almost every game.")

## 3. Play against it
Run `play_human(depth=5)` and type a column (0–6) each turn. You are O.

In [ ]:
def play_human(depth=5):
    board, player = np.zeros((6, 7), int), 1
    while legal_moves(board) and not wins(board, 1) and not wins(board, -1):
        if player == 1: board = drop(board, minimax(board, depth)[1], 1)
        else: show(board); board = drop(board, int(input("your column: ")), -1)
        player = -player
    show(board); print("AI wins" if wins(board, 1) else "You win!" if wins(board, -1) else "Draw")
# play_human()

<details><summary>▶ Solution</summary>

```python
def wins(board, player):
    R, C = board.shape
    for r in range(R):
        for c in range(C):
            if board[r, c] != player:
                continue
            for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                if all(0 <= r + k * dr < R and 0 <= c + k * dc < C and board[r + k * dr, c + k * dc] == player for k in range(4)):
                    return True
    return False
```

```python
def minimax(board, depth, alpha=-math.inf, beta=math.inf, maximizing=True):
    if wins(board, 1): return 1_000_000 + depth, None
    if wins(board, -1): return -1_000_000 - depth, None
    moves = legal_moves(board)
    if depth == 0 or not moves: return heuristic(board), None
    best_move = moves[0]
    if maximizing:
        value = -math.inf
        for m in moves:
            v, _ = minimax(drop(board, m, 1), depth - 1, alpha, beta, False)
            if v > value: value, best_move = v, m
            alpha = max(alpha, value)
            if alpha >= beta: break
    else:
        value = math.inf
        for m in moves:
            v, _ = minimax(drop(board, m, -1), depth - 1, alpha, beta, True)
            if v < value: value, best_move = v, m
            beta = min(beta, value)
            if alpha >= beta: break
    return value, best_move
```
</details>

## Stretch goals
- Count nodes visited with and without pruning (and with/without centre-first move ordering).
- Replace the heuristic with **Monte Carlo tree search**: random playouts instead of a hand-written evaluation (the AlphaGo family).
- Learn the heuristic: play 10,000 self-play games and fit a logistic regression on window counts.